# Train PPO on continuous actions

PPO updates a Gaussian actor using fixed generalized advantage estimates (GAE):

$$L_\pi=-\frac1N\sum_t\min(\rho_t\hat A_t,\operatorname{clip}(\rho_t,1-\epsilon,1+\epsilon)\hat A_t).$$

Here $\rho_t=\pi_\theta(a_t\mid s_t)/\pi_{old}(a_t\mid s_t)$ is the probability ratio, $\hat A_t$ the advantage, and $N$ the rollout size. The clipping width is $\epsilon$. We train on `Pendulum-v1` with lower gravity (`g=1.0`) for a shorter CPU demonstration. A tanh transform and affine rescaling keep actions within the torque bounds. PPO sums log densities over action dimensions before computing each ratio.

In [ ]:
import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np

from aprenderl import PPO, PPOConfig
from aprenderl.utils import evaluate_policy

ENV_ID = "Pendulum-v1"
ENV_KWARGS = {"g": 1.0}

In [ ]:
env = gym.make(ENV_ID, **ENV_KWARGS)
try:
    config = PPOConfig(
        learning_rate=3e-4,
        batch_size=64,
        n_epochs=10,
        value_learning_rate=1e-3,
        n_steps=1024,
        gae_lambda=0.95,
        normalize_advantage=True,
        seed=7,
    )
    agent = PPO(env, config=config, device="cpu")
    agent.learn(total_timesteps=51_200)
finally:
    env.close()

In [ ]:
returns = np.asarray(agent.episode_returns)
window = min(10, len(returns))
moving_average = np.convolve(returns, np.ones(window) / window, mode="valid")

plt.figure(figsize=(8, 4))
plt.plot(returns, alpha=0.35, label="Episode return")
plt.plot(
    np.arange(window - 1, len(returns)),
    moving_average,
    label=f"{window}-episode average",
)
plt.xlabel("Episode")
plt.ylabel("Return")
plt.title(f"PPO training on {ENV_ID}")
plt.legend()
plt.grid(alpha=0.2)
plt.show()

## Watch the trained policy

This opens a window and runs 5 episodes using deterministic actions.

In [ ]:
evaluation_env = gym.make(
    ENV_ID, render_mode="human", **ENV_KWARGS
)
try:
    result = evaluate_policy(
        agent, evaluation_env, episodes=5, deterministic=True, seed=1_000
    )
finally:
    evaluation_env.close()

print("Episode returns:", result.returns)
print(f"Mean return: {result.mean_return:.1f} +/- {result.return_std:.1f}")